# Clinical Note Triage — starter notebook

Read a free-text **medical transcription** and predict which **clinical specialty**
it belongs to — automatic triage of clinical notes, across **12 specialties**
(`spec_01` … `spec_12`).

This notebook runs end-to-end: **download the data → train a simple model → submit**.
Metric: **F1-macro**.

> Open in Colab, then run the cells top to bottom. The only thing you must edit is
> your API token in the next cell (find it on your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" scikit-learn pandas  # installs the `mlarena` package

import mlarena
import pandas as pd

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
COMPETITION_ID = 174

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`download_dataset` pulls the competition files into `data/`. Each row is one free-text
clinical transcription; the training labels are in `train.csv` (`id,transcription,label`).

In [ ]:
client.download_dataset(COMPETITION_ID, "data/")

train = pd.read_csv("data/train.csv")   # id, transcription, label
test  = pd.read_csv("data/test.csv")    # id, transcription

print("train:", train.shape, "  test:", test.shape)
print(train["label"].value_counts())
train.head(2)

## 2. A baseline model

Turn each transcription into **TF-IDF** features (word + bigram counts, down-weighted by
how common they are) and fit a logistic regression. This is just a starting point — word
embeddings or a fine-tuned transformer will score much higher. We use 3-fold
cross-validation to estimate the F1-macro before submitting.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

model = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                              min_df=3, max_features=40000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced")),
])

cv = cross_val_score(model, train["transcription"], train["label"], cv=3, scoring="f1_macro")
print("CV F1-macro: %.4f +/- %.4f" % (cv.mean(), cv.std()))

model.fit(train["transcription"], train["label"])   # refit on all the training data

## 3. Predict and write `submission.csv`

In [ ]:
pred = model.predict(test["transcription"])
submission = pd.DataFrame({"id": test["id"], "label": pred})
submission.to_csv("submission.csv", index=False)
submission.head()

## 4. Submit

Run the cell below to submit through the SDK, or upload `submission.csv` on the
competition page.

In [ ]:
result = client.submit(competition_id=COMPETITION_ID, files=["submission.csv"])
print(result)

## 5. Where to go from here

TF-IDF + logistic regression is a strong, fast baseline — but it only sees *words*, not
*meaning*. Work one step at a time: **change one thing → check the CV F1 → submit only if it
improved.** Trust the cross-validated number, not a single split.

- **Read the data first.** Look at a few notes per specialty; note their length, vocabulary,
  and class balance. F1-macro weights every specialty equally, so the rare ones matter most —
  watch how the baseline does on each (`classification_report`).

- **Squeeze more from the baseline.** Tune the TF-IDF (`ngram_range`, `min_df`, `max_features`)
  and the classifier's `C`, or swap in linear SVM / a gradient-boosting model. Cheap wins
  before you reach for anything heavy.

- **Pick a model that fits the domain.** Browse Hugging Face for one of the right **size** and
  trained on the right **corpus** — clinical / biomedical text (e.g. a BioBERT / ClinicalBERT
  variant) understands medical vocabulary a general model misses.

- **Try it without training.** **Zero-shot** and **few-shot** classification (a pretrained LLM,
  or `zero-shot-classification` pipeline) give a baseline with no fitting at all — useful to
  see how far raw language understanding gets you.

- **Use embeddings.** Turn each note into a sentence embedding, **visualize** them (UMAP / t-SNE
  — do the specialties cluster?), then train a simple classifier on top. Often most of the
  transformer's gain for a fraction of the cost.

- **Fine-tune.** Adapt a transformer end-to-end to this task — **LoRA / PEFT** makes it cheap
  enough for a single GPU.

Keep cross-validation as your compass, and mind the input length limit — long notes may need
truncation or chunking.
